# 04 · Execution, runtime guardrails and step records

**Role of this layer.** The executor is the only code that calls capability
functions. It re-validates the plan (it does not trust its caller), refuses to
start while any BLOCK exists or any APPROVAL issue is unapproved, then runs the
steps in order with the runtime guardrails that need data:

| code | severity | checks |
|---|---|---|
| R01 | BLOCK | snapshot file hash still matches the catalog |
| R02 | BLOCK | enough observations for the capability (e.g. 3 x window) |
| R03 | WARN | more than 5% of a raw series missing |
| R04 | WARN | an inner join drops more than 10% of an input |

Every step produces a **step record**: parameters used (defaults filled), input
hashes, output hash, sample size and range, duration, issues.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.data import load_catalog
from quantlab.executor import execute_plan
from quantlab.plan import Plan
from quantlab.scenarios import SCENARIOS_BY_ID
from quantlab.validation import fill_defaults

catalog = load_catalog(settings.catalog_path)


def scenario(scenario_id):
    s = SCENARIOS_BY_ID[scenario_id]
    return fill_defaults(Plan(plan_id="demo", request=s.request, **s.plan))


plan_a = scenario("A")
refused = execute_plan(plan_a, catalog, approved_issue_ids=set())
print(refused.status, "-", refused.error)

blocked - Approval required for: G12:s2:return_method


With the approval granted, scenario A runs on the real S&P 500 snapshot:

In [3]:
result = execute_plan(plan_a, catalog, approved_issue_ids={"G12:s2:return_method"})
print("status:", result.status, f"({result.duration_ms:.0f} ms)")
pd.DataFrame(
    [
        {
            "step": r.step_id,
            "capability": r.capability,
            "params": r.params,
            "n_obs": r.n_obs,
            "range": f"{r.start}..{r.end}",
            "ms": r.duration_ms,
            "output sha256": r.output_hash[:12],
        }
        for r in result.step_records
    ]
)

status: completed (21 ms)


,step,capability,params,n_obs,range,ms,output sha256
0,s1,load_series,{'dataset_id': 'SP500'},2513,2016-10-17..2026-10-15,11.92,bcd4dff9bbcd
1,s2,compute_returns,"{'method': 'simple', 'periods': 1}",2512,2016-10-18..2026-10-15,8.67,b3b5b3a25063
2,s3,summary_stats,{},2512,2016-10-18..2026-10-15,0.26,d6b3519254d6


The last step is the summary table; its `mean` is the average daily return.

In [4]:
pd.Series(result.artifacts["s3"].value)

count                    2512
mean                  0.00058
std                  0.011383
min                 -0.119841
max                  0.095154
skew                -0.369907
excess_kurtosis     15.916102
start              2016-10-18
end                2026-10-15
dtype: object

**R02**: a 250-month window on monthly data does not have 3 x 250 observations.

In [5]:
from quantlab.plan import Step

plan = fill_defaults(
    Plan(
        plan_id="demo",
        request="monthly volatility",
        dataset_ids=["NASDAQCOM"],
        steps=[
            Step(id="s1", capability="load_series", params={"dataset_id": "NASDAQCOM"}),
            Step(id="s2", capability="resample", inputs={"series": "s1"}, params={"freq": "M"}),
            Step(
                id="s3",
                capability="compute_returns",
                inputs={"series": "s2"},
                params={"method": "simple"},
            ),
            Step(
                id="s4",
                capability="rolling_volatility",
                inputs={"series": "s3"},
                params={"window": 250},
            ),
        ],
    )
)
blocked = execute_plan(plan, catalog, {"G12:s3:return_method"})
print(blocked.status, "-", blocked.error)

blocked - Step s4 (rolling_volatility): rolling_volatility has 667 usable observations; it needs 750 (at least 3 x window observations).


**R04** fires on the real data in scenario E: the 2-year yield starts in 1976 and
the 10-year in 1962, so the spread silently loses 22% of the 10-year history.
The run completes, with the warning recorded.

In [6]:
result_e = execute_plan(scenario("E"), catalog)
print(result_e.status)
for issue in result_e.issues:
    print(f"[{issue.severity}] {issue.code}: {issue.message}")

completed
[WARN] R04: Aligning the inputs kept 12587 of 16179 rows of 'a' (22% dropped).


Determinism: running the same plan twice gives identical output hashes.

In [7]:
again = execute_plan(scenario("E"), catalog)
[r.output_hash for r in again.step_records] == [r.output_hash for r in result_e.step_records]

True